# Setup CLI

> The `claude-plugins setup` command — one-shot dev environment bootstrapper

In [ ]:
#| default_exp setup_cmd

In [ ]:
#| export
from __future__ import annotations
import argparse
import json
import shutil
import subprocess
import sys
from pathlib import Path

In [ ]:
#| export
def check_uv() -> bool:
    "Return True if uv is available in PATH."
    return shutil.which('uv') is not None

In [ ]:
#| export
def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    "Run a subprocess command and return the result."
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=False)

In [ ]:
#| export
DEV_PACKAGES = [
    'python-fasthtml',
    'MonsterUI',
    'litesearch',
    'lisette',
    'exhash',
    'safecmd',
    'safepyrun',
    'codesigs',
    'nbdev',
    'jupyter',
    'ipykernel',
    'mcp',
    'claude-plugins',
]

In [ ]:
#| export
DEFAULT_SAFECMD_ALLOWLIST = {
    "allowed_commands": [
        "uv", "pip", "pip3", "pipx",
        "python", "python3", "ipython",
        "nbdev-export", "nbdev-test", "nbdev-docs", "nbdev-prepare",
        "nbdev-clean", "nbdev-install-hooks", "nbdev-new",
        "git",
        "pytest", "coverage",
        "jupyter", "nbconvert",
        "ls", "cat", "head", "tail", "grep", "find", "wc", "sort", "uniq",
        "mkdir", "touch", "cp", "mv",
        "claude",
        "make", "cargo",
        "echo", "printf", "sed", "awk", "cut", "tr",
        "curl", "wget",
    ],
    "_comment": "Edit this list to customise which commands Claude Code can run via Bash."
}

In [ ]:
#| export
HOOKS_CONFIG = {
    "hooks": [
        {
            "event": "PreToolUse",
            "matcher": "Bash",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.safecmd_hook"
            }
        },
        {
            "event": "PostToolUse",
            "matcher": "Edit|Write",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.exhash_hook"
            }
        },
        {
            "event": "SessionStart",
            "action": {
                "type": "command",
                "command": "uv run python -m claude_plugins.hooks.index_hook"
            }
        }
    ]
}

MCP_CONFIG = {
    "mcpServers": {
        "exhash": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_exhash"]
        },
        "safecmd": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_safecmd"]
        },
        "safepyrun": {
            "type": "stdio",
            "command": "uv",
            "args": ["run", "python", "-m", "claude_plugins.mcp_safepyrun"]
        }
    }
}

In [ ]:
#| export
CLAUDE_MD_BLOCK = '''
<!-- claude-plugins: START -->
## Claude Code Setup (claude-plugins)

Active hooks: safecmd (Bash PreToolUse), exhash (Edit/Write PostToolUse), code indexer (SessionStart).
MCP servers available: exhash, safecmd, safepyrun.
Skills: /exhash /safecmd /safepyrun /litesearch /fasthtml /monsterui /lisette /nbdev /codesigs
Code index: .claude/code_index.db (built on session start via codesigs + litesearch)
<!-- claude-plugins: END -->
'''

In [ ]:
#| export
def upsert_claude_md(target: Path) -> None:
    """Add or update the claude-plugins block in CLAUDE.md."""
    claude_md = target / 'CLAUDE.md'
    if claude_md.exists():
        content = claude_md.read_text()
        # Remove old block if present
        if '<!-- claude-plugins: START -->' in content:
            import re
            content = re.sub(
                r'<!-- claude-plugins: START -->.*?<!-- claude-plugins: END -->',
                '', content, flags=re.DOTALL
            ).strip()
        claude_md.write_text(content + '\n' + CLAUDE_MD_BLOCK)
    else:
        claude_md.write_text(CLAUDE_MD_BLOCK)

In [ ]:
#| export
def write_json(path: Path, data: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2) + '\n')


def merge_json_file(path: Path, new_data: dict) -> None:
    """Merge new_data into existing JSON at path; lists are extended without duplicates."""
    from claude_plugins.core import merge_json
    merge_json(path, new_data)

In [ ]:
#| export
def install_packages_uv(target: Path) -> None:
    """Install dev packages into the target project using UV."""
    print('\n[2/8] Installing packages with UV...')
    # Add all packages as optional/dev dependencies
    for pkg in DEV_PACKAGES:
        print(f'      uv add --optional dev {pkg}')
        try:
            run(['uv', 'add', '--optional', 'dev', pkg], cwd=target, check=False)
        except Exception as e:
            print(f'      WARNING: failed to add {pkg}: {e}')
    # Sync everything
    run(['uv', 'sync', '--extra', 'dev'], cwd=target, check=False)

In [ ]:
#| export
def setup(target_dir: str | None = None, skip_packages: bool = False) -> None:
    """Run the full claude-plugins setup in target_dir (default: cwd)."""
    target = Path(target_dir or Path.cwd()).resolve()
    print(f'claude-plugins setup — target: {target}')
    print('=' * 60)

    # Step 1: Check UV
    print('\n[1/8] Checking UV...')
    if not check_uv():
        print('  ERROR: UV not found. Install it from https://docs.astral.sh/uv/getting-started/installation/')
        print('  curl -LsSf https://astral.sh/uv/install.sh | sh')
        sys.exit(1)
    print('  ✓ UV found')

    # Step 2: Install packages
    if not skip_packages:
        install_packages_uv(target)
        print('  ✓ Packages installed')

    # Step 3: nbdev-install-hooks
    print('\n[3/8] Installing nbdev hooks (git + Jupyter)...')
    try:
        run(['uv', 'run', 'nbdev-install-hooks'], cwd=target, check=False)
        print('  ✓ nbdev hooks installed')
    except Exception as e:
        print(f'  WARNING: nbdev-install-hooks failed: {e}')

    # Step 4: Claude Code hooks
    print('\n[4/8] Writing .claude/settings.json (hooks)...')
    settings_path = target / '.claude' / 'settings.json'
    merge_json_file(settings_path, HOOKS_CONFIG)
    print(f'  ✓ {settings_path}')

    # Step 5: MCP servers
    print('\n[5/8] Writing .mcp.json (MCP servers)...')
    mcp_path = target / '.mcp.json'
    merge_json_file(mcp_path, MCP_CONFIG)
    print(f'  ✓ {mcp_path}')

    # Step 6: safecmd allowlist
    print('\n[6/8] Writing .claude/safecmd_allowlist.json...')
    allowlist_path = target / '.claude' / 'safecmd_allowlist.json'
    if not allowlist_path.exists():
        write_json(allowlist_path, DEFAULT_SAFECMD_ALLOWLIST)
        print(f'  ✓ {allowlist_path}')
    else:
        print(f'  (skipped — already exists: {allowlist_path})')

    # Step 7: Skills
    print('\n[7/8] Installing skills to .claude/skills/...')
    from claude_plugins.skills_gen import install_skills
    installed = install_skills(target)
    for name in installed:
        print(f'  ✓ {name}')

    # Step 8: CLAUDE.md
    print('\n[8/8] Updating CLAUDE.md...')
    upsert_claude_md(target)
    print(f'  ✓ {target / "CLAUDE.md"}')

    print('\n' + '=' * 60)
    print('claude-plugins setup complete!')
    print('\nNext steps:')
    print('  1. Start a Claude Code session: claude')
    print('  2. Try: /exhash, /safecmd, /safepyrun, /litesearch')
    print('  3. Review .claude/safecmd_allowlist.json to customise allowed commands')
    print('  4. See CLAUDE.md for full usage guide')

In [ ]:
#| export
def main():
    parser = argparse.ArgumentParser(
        prog='claude-plugins',
        description='Claude Code dev environment setup — hooks, MCP servers, and skills',
    )
    subparsers = parser.add_subparsers(dest='command')

    setup_parser = subparsers.add_parser('setup', help='Run full setup in target directory')
    setup_parser.add_argument(
        '--target-dir', '-t', default=None,
        help='Target project directory (default: current directory)',
    )
    setup_parser.add_argument(
        '--skip-packages', action='store_true',
        help='Skip UV package installation (hooks, MCP, skills only)',
    )

    args = parser.parse_args()

    if args.command == 'setup':
        setup(target_dir=args.target_dir, skip_packages=args.skip_packages)
    else:
        parser.print_help()


if __name__ == '__main__':
    main()